# Training Notebook — Vision Transformer on MNIST
## Investigating Gradient Flow via Orthogonal Initialization

**Author:** Nikolaos Mavros — University of Thessaly, ECE452  
**Repo:** `vit-gradient-flow`

This notebook trains two lightweight ViT models on MNIST and compares:
1. **Xavier/Glorot** initialization (baseline)
2. **Orthogonal** initialization ($W^TW = I$, experimental)

For each, we track: loss, accuracy, gradient norms per layer, and Jacobian condition numbers through the attention blocks.

**Target hardware:** ThinkPad 2024 — Ryzen 5, 16 GB RAM, CPU-only.

## 1. Environment Setup

Make sure you're running this from the repo root where `Project.toml` lives.  
The cell below activates the project environment and installs any missing packages.

In [3]:
using Pkg
Pkg.activate("..")   # repo root (one level up from notebooks/)
Pkg.instantiate()

using Flux
using Flux: onehotbatch, onecold, logitcrossentropy, DataLoader
using Zygote
using MLDatasets
using LinearAlgebra
using Statistics
using Random
using JLD2
using ProgressMeter

println("All packages loaded ✓")

  Activating project at `c:\Users\nickb\Documents\vit-gradient-flow`
    Updating registry at `C:\Users\nickb\.julia\registries\General.toml`
   Installed ChainRulesCore ─ v1.26.1
   Installed PrettyTables ─── v3.3.2
    Updating `C:\Users\nickb\Documents\vit-gradient-flow\Project.toml`
  [587475ba] + Flux v0.16.9
  [033835bb] + JLD2 v0.6.4
  [eb30cadb] + MLDatasets v0.7.21
  [92933f4c] + ProgressMeter v1.11.0
  [10745b16] + Statistics v1.11.1
  [e88e6eb3] + Zygote v0.7.10
  [37e2e46d] ~ LinearAlgebra ⇒ v1.12.0
  [9a3f8284] ~ Random ⇒ v1.11.0
    Updating `C:\Users\nickb\Documents\vit-gradient-flow\Manifest.toml`
  [47edcb42] + ADTypes v1.21.0
  [621f4979] + AbstractFFTs v1.5.0
  [7d9f7c33] + Accessors v0.1.44
  [79e6a3ab] + Adapt v4.5.0
  [66dad0bd] + AliasTables v1.1.3
  [dce04be8] + ArgCheck v2.5.0
  [a9b6321e] + Atomix v1.1.3
  [a963bdd2] + AtomsBase v0.5.2
⌅ [ab4f0b2a] + BFloat16s v0.5.1
  [198e06fe] + BangBang v0.4.9
  [9718e550] + Baselet v0.1.1
  [d1d4a3ce] + BitFlags v0.1.9
  

All packages loaded ✓

  11605.8 ms  ✓ MLDatasets
  33 dependencies successfully precompiled in 114 seconds. 245 already precompiled.


## 2. Hyperparameters

These are tuned to be lightweight enough for CPU training on your ThinkPad.  
With `D_MODEL=64`, `NUM_BLOCKS=2`, and 16 patches, the model has ~120K parameters.  
Training 15 epochs takes roughly 15–25 minutes on a Ryzen 5.

In [4]:
# ── Image & Patch geometry ────────────────────────────────────────────────
const IMG_SIZE      = 28          # MNIST 28×28
const PATCH_SIZE    = 7           # 7×7 patches → 4×4 grid
const NUM_PATCHES   = (IMG_SIZE ÷ PATCH_SIZE)^2   # 16
const PATCH_DIM     = PATCH_SIZE * PATCH_SIZE      # 49

# ── Transformer architecture ─────────────────────────────────────────────
const D_MODEL       = 64          # embedding dim
const NUM_HEADS     = 4           # attention heads
const HEAD_DIM      = D_MODEL ÷ NUM_HEADS          # 16
const MLP_HIDDEN    = 128         # FFN hidden dim
const NUM_BLOCKS    = 2           # encoder blocks
const NUM_CLASSES   = 10          # digits 0–9

# ── Training ─────────────────────────────────────────────────────────────
const BATCH_SIZE    = 128
const EPOCHS        = 15
const LR            = 1e-3
const WEIGHT_DECAY  = 1e-4

# ── Diagnostics ──────────────────────────────────────────────────────────
const JACOBIAN_EVERY = 5          # compute Jacobian κ every N epochs

# ── Output directory ─────────────────────────────────────────────────────
const OUT_DIR = joinpath("..", "results")
mkpath(joinpath(OUT_DIR, "figures"))
mkpath(joinpath(OUT_DIR, "logs"))

println("Hyperparameters set ✓")
println("  Patches per image: $NUM_PATCHES  |  Patch dim: $PATCH_DIM")
println("  Model dim: $D_MODEL  |  Heads: $NUM_HEADS  |  Blocks: $NUM_BLOCKS")

Hyperparameters set ✓
  Patches per image: 16  |  Patch dim: 49
  Model dim: 64  |  Heads: 4  |  Blocks: 2


## 3. Initialization Strategies

Two strategies from the proposal:

- **Xavier/Glorot**: scales by $\sqrt{6 / (\text{fan\_in} + \text{fan\_out})}$ — standard default.
- **Orthogonal**: uses QR decomposition to produce a matrix with all singular values = 1.  
  This is the key hypothesis: orthogonal weights should preserve gradient norms through layers, yielding Jacobian condition numbers closer to 1.

In [5]:
function xavier_init(rng::AbstractRNG, dims...)
    fan_in  = dims[end-1] isa Integer ? dims[end-1] : prod(dims[1:end-1])
    fan_out = dims[end]
    limit   = Float32(sqrt(6.0 / (fan_in + fan_out)))
    return (rand(rng, Float32, dims...) .- 0.5f0) .* 2.0f0 .* limit
end
xavier_init(dims...) = xavier_init(Random.default_rng(), dims...)

function orthogonal_init(rng::AbstractRNG, dims...)
    rows, cols = dims[1], dims[2]
    n = max(rows, cols)
    A = randn(rng, Float32, n, n)
    Q, _ = qr(A)
    Q_mat = Matrix{Float32}(Q)
    return Q_mat[1:rows, 1:cols]
end
orthogonal_init(dims...) = orthogonal_init(Random.default_rng(), dims...)

# Quick sanity check — orthogonal matrix should have singular values ≈ 1
W_test = orthogonal_init(64, 64)
sv = svdvals(W_test)
println("Orthogonal init sanity check:")
println("  Singular values range: [$(round(minimum(sv); digits=4)), $(round(maximum(sv); digits=4))]")
println("  WᵀW ≈ I check (Frobenius norm of WᵀW - I): $(round(norm(W_test' * W_test - I); digits=6))")

Orthogonal init sanity check:
  Singular values range: [1.0, 1.0]
  WᵀW ≈ I check (Frobenius norm of WᵀW - I): 3.0e-6


## 4. Model Components

All layers are built manually (not using Flux's built-in `MultiHeadAttention`) so we have full access to `WQ`, `WK`, `WV`, `WO` for gradient inspection and Jacobian analysis.

### 4.1 Patch Embedding

Flattened 7×7 patches are linearly projected into `D_MODEL` dimensions.  
A learnable CLS token is prepended, and learnable positional embeddings are added.

In [15]:
struct PatchEmbedding
    proj::Dense                               # PATCH_DIM → D_MODEL
    cls_token::AbstractArray{Float32, 2}      # (D_MODEL, 1)
    pos_embed::AbstractArray{Float32, 2}      # (D_MODEL, NUM_PATCHES + 1)
end

Flux.@layer PatchEmbedding

function PatchEmbedding(init_fn)
    # Flux Dense(weight, bias): weight must be (out, in) = (D_MODEL, PATCH_DIM)
    proj = Dense(init_fn(D_MODEL, PATCH_DIM), zeros(Float32, D_MODEL))
    cls  = randn(Float32, D_MODEL, 1) .* 0.02f0
    pos  = randn(Float32, D_MODEL, NUM_PATCHES + 1) .* 0.02f0
    return PatchEmbedding(proj, cls, pos)
end

function (pe::PatchEmbedding)(patches)
    # patches: (PATCH_DIM, NUM_PATCHES, batch)
    B = size(patches, 3)
    x = pe.proj(reshape(patches, PATCH_DIM, :))          # (D_MODEL, NUM_PATCHES*B)
    x = reshape(x, D_MODEL, NUM_PATCHES, B)              # (D_MODEL, NUM_PATCHES, B)
    cls_expanded = repeat(pe.cls_token, 1, 1, B)          # (D_MODEL, 1, B)
    x = cat(cls_expanded, x; dims=2)                      # (D_MODEL, NUM_PATCHES+1, B)
    x = x .+ pe.pos_embed                                 # broadcast over batch
    return x
end

println("PatchEmbedding defined ✓")

PatchEmbedding defined ✓


### 4.2 Multi-Head Self-Attention

Implements scaled dot-product attention:

$$\text{Attn}(X) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

with separate `WQ`, `WK`, `WV`, `WO` weight matrices — exactly as described in the proposal's Equation (3).

In [16]:
struct MultiHeadAttention
    wq::Dense
    wk::Dense
    wv::Dense
    wo::Dense
end

Flux.@layer MultiHeadAttention

function MultiHeadAttention(init_fn)
    # Dense(weight, bias): weight is (out, in) = (D_MODEL, D_MODEL)
    wq = Dense(init_fn(D_MODEL, D_MODEL), zeros(Float32, D_MODEL))
    wk = Dense(init_fn(D_MODEL, D_MODEL), zeros(Float32, D_MODEL))
    wv = Dense(init_fn(D_MODEL, D_MODEL), zeros(Float32, D_MODEL))
    wo = Dense(init_fn(D_MODEL, D_MODEL), zeros(Float32, D_MODEL))
    return MultiHeadAttention(wq, wk, wv, wo)
end

function (mha::MultiHeadAttention)(x)
    # x: (D_MODEL, seq_len, batch)
    D, S, B = size(x)

    # Linear projections
    Q = mha.wq(reshape(x, D, :))    # (D_MODEL, S*B)
    K = mha.wk(reshape(x, D, :))
    V = mha.wv(reshape(x, D, :))

    # Reshape to multi-head: (HEAD_DIM, S, NUM_HEADS, B)
    Q = permutedims(reshape(Q, HEAD_DIM, NUM_HEADS, S, B), (1, 3, 2, 4))
    K = permutedims(reshape(K, HEAD_DIM, NUM_HEADS, S, B), (1, 3, 2, 4))
    V = permutedims(reshape(V, HEAD_DIM, NUM_HEADS, S, B), (1, 3, 2, 4))

    # Merge head+batch dims for batched matmul
    Q_flat = reshape(Q, HEAD_DIM, S, NUM_HEADS * B)
    K_flat = reshape(K, HEAD_DIM, S, NUM_HEADS * B)
    V_flat = reshape(V, HEAD_DIM, S, NUM_HEADS * B)

    # Scaled dot-product: (S, S, H*B)
    scale = Float32(1.0 / sqrt(HEAD_DIM))
    scores = Flux.batched_mul(
        permutedims(Q_flat, (2, 1, 3)),    # (S, HEAD_DIM, H*B)
        K_flat                              # (HEAD_DIM, S, H*B)
    ) .* scale

    attn_weights = softmax(scores; dims=2)  # (S, S, H*B)

    # Weighted sum: (HEAD_DIM, S, H*B)
    out = Flux.batched_mul(V_flat, permutedims(attn_weights, (2, 1, 3)))

    # Reshape back: (D_MODEL, S, B)
    out = reshape(permutedims(reshape(out, HEAD_DIM, S, NUM_HEADS, B), (1, 3, 2, 4)),
                  D_MODEL, S, B)

    # Output projection
    out_flat = mha.wo(reshape(out, D_MODEL, :))
    return reshape(out_flat, D_MODEL, S, B)
end

println("MultiHeadAttention defined ✓")

MultiHeadAttention defined ✓


### 4.3 Feed-Forward Network & Transformer Block

Each block uses **pre-norm** residual connections:

```
x = x + Attn(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

In [17]:
# ── Feed-Forward Network ──────────────────────────────────────────────────
struct FeedForward
    fc1::Dense
    fc2::Dense
end

Flux.@layer FeedForward

function FeedForward(init_fn)
    # fc1: (MLP_HIDDEN, D_MODEL),  fc2: (D_MODEL, MLP_HIDDEN)
    fc1 = Dense(init_fn(MLP_HIDDEN, D_MODEL), zeros(Float32, MLP_HIDDEN), gelu)
    fc2 = Dense(init_fn(D_MODEL, MLP_HIDDEN), zeros(Float32, D_MODEL))
    return FeedForward(fc1, fc2)
end

function (ff::FeedForward)(x)
    D, S, B = size(x)
    h = ff.fc1(reshape(x, D, :))
    h = ff.fc2(h)
    return reshape(h, D, S, B)
end

# ── Transformer Block ────────────────────────────────────────────────────
struct TransformerBlock
    ln1::LayerNorm
    attn::MultiHeadAttention
    ln2::LayerNorm
    ff::FeedForward
end

Flux.@layer TransformerBlock

function TransformerBlock(init_fn)
    return TransformerBlock(
        LayerNorm(D_MODEL),
        MultiHeadAttention(init_fn),
        LayerNorm(D_MODEL),
        FeedForward(init_fn)
    )
end

function (tb::TransformerBlock)(x)
    D, S, B = size(x)

    # Self-attention + residual
    x_norm = reshape(tb.ln1(reshape(x, D, :)), D, S, B)
    x = x .+ tb.attn(x_norm)

    # FFN + residual
    x_norm2 = reshape(tb.ln2(reshape(x, D, :)), D, S, B)
    x = x .+ tb.ff(x_norm2)

    return x
end

println("FeedForward & TransformerBlock defined ✓")

FeedForward & TransformerBlock defined ✓


### 4.4 Full Vision Transformer

Combines: PatchEmbedding → N × TransformerBlock → LayerNorm → Classification Head.  
The CLS token (position 1) is used for classification, following the original ViT design.

In [18]:
struct ViT
    patch_embed::PatchEmbedding
    blocks::Vector{TransformerBlock}
    ln_final::LayerNorm
    head::Dense
end

Flux.@layer ViT

function ViT(init_fn)
    pe     = PatchEmbedding(init_fn)
    blocks = [TransformerBlock(init_fn) for _ in 1:NUM_BLOCKS]
    ln     = LayerNorm(D_MODEL)
    # head: (NUM_CLASSES, D_MODEL)
    head   = Dense(init_fn(NUM_CLASSES, D_MODEL), zeros(Float32, NUM_CLASSES))
    return ViT(pe, blocks, ln, head)
end

function (model::ViT)(patches)
    # patches: (PATCH_DIM, NUM_PATCHES, batch)
    x = model.patch_embed(patches)           # (D_MODEL, NUM_PATCHES+1, B)
    for block in model.blocks
        x = block(x)
    end
    cls = x[:, 1, :]                         # CLS token → (D_MODEL, B)
    cls = model.ln_final(cls)
    logits = model.head(cls)                 # (NUM_CLASSES, B)
    return logits
end

println("ViT defined ✓")

ViT defined ✓


## 5. Data Loading — MNIST as Patch Sequences

MNIST images (28×28) are split into a 4×4 grid of 7×7 patches, each flattened to a 49-dim vector. This converts each image into a sequence of 16 tokens — the input format for the ViT.

In [19]:
function images_to_patches(images)
    # images: (28, 28, batch) Float32
    B = size(images, 3)
    n_per_side = IMG_SIZE ÷ PATCH_SIZE
    patches = zeros(Float32, PATCH_DIM, NUM_PATCHES, B)

    idx = 1
    for row in 1:n_per_side
        for col in 1:n_per_side
            r_s = (row - 1) * PATCH_SIZE + 1
            c_s = (col - 1) * PATCH_SIZE + 1
            patch = @view images[r_s:r_s+PATCH_SIZE-1, c_s:c_s+PATCH_SIZE-1, :]
            patches[:, idx, :] .= reshape(patch, PATCH_DIM, B)
            idx += 1
        end
    end
    return patches
end

# ── Load MNIST ───────────────────────────────────────────────────────────
train_x, train_y = MLDatasets.MNIST(split=:train)[:]
test_x,  test_y  = MLDatasets.MNIST(split=:test)[:]

train_x = Float32.(train_x)
test_x  = Float32.(test_x)

train_labels = onehotbatch(train_y, 0:9)
test_labels  = onehotbatch(test_y,  0:9)

train_patches = images_to_patches(train_x)
test_patches  = images_to_patches(test_x)

train_loader = DataLoader((train_patches, train_labels);
                           batchsize=BATCH_SIZE, shuffle=true, partial=false)
test_loader  = DataLoader((test_patches, test_labels);
                           batchsize=BATCH_SIZE, shuffle=false, partial=true)

println("Data loaded ✓")
println("  Train: $(size(train_patches, 3)) images → patches $(size(train_patches))")
println("  Test:  $(size(test_patches, 3)) images → patches $(size(test_patches))")

Data loaded ✓
  Train: 60000 images → patches (49, 16, 60000)
  Test:  10000 images → patches (49, 16, 10000)


## 6. Gradient Flow Diagnostics

Two diagnostic tools from the proposal:

1. **Gradient norm tracking** — records $\|\nabla_{W}\mathcal{L}\|_2$ for every trainable parameter each epoch. Vanishing → norms collapse to 0 in early layers; exploding → norms blow up.

2. **Jacobian condition number** — for a single sample, computes the Jacobian of each transformer block's output w.r.t. its input, then $\kappa(J) = \sigma_{\max} / \sigma_{\min}$. Orthogonal init should yield $\kappa \approx 1$.

In [23]:
function collect_gradient_norms(grads, model)
    norms = Dict{String, Float64}()
    # grads is a nested NamedTuple mirroring the model structure
    # We flatten it recursively to get (name => gradient_array) pairs
    function _walk(prefix, g)
        if g isa AbstractArray
            norms[prefix] = Float64(norm(vec(g)))
        elseif g isa NamedTuple
            for k in keys(g)
                _walk(prefix == "" ? string(k) : prefix * "." * string(k), g[k])
            end
        elseif g isa AbstractVector   # e.g. blocks::Vector{NamedTuple}
            for (i, item) in enumerate(g)
                _walk(prefix * "[$i]", item)
            end
        end
        # skip Nothing, numbers, etc.
    end
    _walk("", grads)
    return norms
end

function compute_jacobian_condition(model, sample_patches)
    # sample_patches: (PATCH_DIM, NUM_PATCHES, 1)
    cond_numbers = Float64[]

    x = model.patch_embed(sample_patches)

    for (i, block) in enumerate(model.blocks)
        x_vec = vec(x)
        J = Zygote.jacobian(z -> vec(block(reshape(z, size(x)))), x_vec)[1]

        if J !== nothing && !isempty(J)
            sv = svdvals(J)
            sv_pos = filter(s -> s > 1e-10, sv)
            if length(sv_pos) >= 2
                push!(cond_numbers, Float64(sv_pos[1] / sv_pos[end]))
            else
                push!(cond_numbers, NaN)
            end
        else
            push!(cond_numbers, NaN)
        end

        x = block(x)   # advance to next block
    end
    return cond_numbers
end

function accuracy(model, loader)
    correct = 0; total = 0
    for (x, y) in loader
        ŷ = model(x)
        correct += sum(onecold(ŷ) .== onecold(y))
        total   += size(y, 2)
    end
    return correct / total
end

println("Diagnostics defined ✓")

Diagnostics defined ✓


## 7. Training Loop

Trains a ViT model with AdamW and logs all metrics. Saves weights + logs as `.jld2` files.

In [24]:
function train_model!(model, train_loader, test_loader, tag::String)
    opt_state = Flux.setup(AdamW(LR, (0.9, 0.999), WEIGHT_DECAY), model)

    log = Dict{String, Vector}(
        "epoch"          => Int[],
        "train_loss"     => Float64[],
        "train_acc"      => Float64[],
        "test_acc"       => Float64[],
        "grad_norms"     => Dict{String, Float64}[],
        "jacobian_conds" => Vector{Float64}[],
    )

    # Fixed sample for Jacobian analysis
    sample_x, _ = first(train_loader)
    jac_sample   = sample_x[:, :, 1:1]

    for epoch in 1:EPOCHS
        epoch_loss  = 0.0
        num_batches = 0
        last_grad_norms = Dict{String, Float64}()

        @info "[$tag] Epoch $epoch / $EPOCHS"

        for (x, y) in train_loader
            loss_val, grads = Flux.withgradient(model) do m
                logitcrossentropy(m(x), y)
            end
            Flux.update!(opt_state, model, grads[1])

            epoch_loss  += loss_val
            num_batches += 1
            last_grad_norms = collect_gradient_norms(grads[1], model)
        end

        avg_loss  = epoch_loss / num_batches
        train_acc = accuracy(model, train_loader)
        test_acc  = accuracy(model, test_loader)

        # Jacobian condition numbers (expensive — periodic)
        jac_conds = Float64[]
        if epoch == 1 || epoch % JACOBIAN_EVERY == 0 || epoch == EPOCHS
            @info "  Computing Jacobian condition numbers …"
            jac_conds = compute_jacobian_condition(model, jac_sample)
            for (i, κ) in enumerate(jac_conds)
                @info "    Block $i κ = $(round(κ; digits=2))"
            end
        end

        @info "  Loss: $(round(avg_loss; digits=4))  " *
              "Train: $(round(100*train_acc; digits=2))%  " *
              "Test: $(round(100*test_acc; digits=2))%"

        push!(log["epoch"],          epoch)
        push!(log["train_loss"],     avg_loss)
        push!(log["train_acc"],      train_acc)
        push!(log["test_acc"],       test_acc)
        push!(log["grad_norms"],     last_grad_norms)
        push!(log["jacobian_conds"], jac_conds)
    end

    # Save weights
    weight_path = joinpath(OUT_DIR, "vit_weights_$(tag).jld2")
    model_state = Flux.state(model)
    @save weight_path model_state
    @info "[$tag] Weights → $weight_path"

    # Save log
    log_path = joinpath(OUT_DIR, "logs", "training_log_$(tag).jld2")
    @save log_path log
    @info "[$tag] Log → $log_path"

    return log
end

println("Training function defined ✓")

Training function defined ✓


## 8. Experiment 1 — Xavier/Glorot Initialization (Baseline)

In [25]:
Random.seed!(42)
model_xavier = ViT(xavier_init)
n_params = sum(length(p) for p in Flux.trainables(model_xavier))
println("ViT (Xavier) — $n_params parameters")

log_xavier = train_model!(model_xavier, train_loader, test_loader, "xavier")

ViT (Xavier) — 72074 parameters


┌ Info: [xavier] Epoch 1 / 15
└ @ Main c:\Users\nickb\Documents\vit-gradient-flow\notebooks\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X26sZmlsZQ==.jl:22


MethodError: MethodError: no method matching iterate(::Nothing)
The function `iterate` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  iterate(!Matched::IRTools.Inner.Pipe)
   @ IRTools C:\Users\nickb\.julia\packages\IRTools\v0mn8\src\ir\ir.jl:889
  iterate(!Matched::IRTools.Inner.Pipe, !Matched::Any)
   @ IRTools C:\Users\nickb\.julia\packages\IRTools\v0mn8\src\ir\ir.jl:889
  iterate(!Matched::Tables.DictRowTable)
   @ Tables C:\Users\nickb\.julia\packages\Tables\cRTb7\src\dicts.jl:122
  ...


## 9. Experiment 2 — Orthogonal Initialization ($W^TW = I$)

In [ ]:
Random.seed!(42)
model_ortho = ViT(orthogonal_init)
println("ViT (Orthogonal) — $(sum(length(p) for p in Flux.trainables(model_ortho))) parameters")

log_ortho = train_model!(model_ortho, train_loader, test_loader, "orthogonal")

## 10. Results Summary

In [ ]:
println("="^70)
println("  TRAINING SUMMARY")
println("="^70)
println()

for (tag, log) in [("Xavier", log_xavier), ("Orthogonal", log_ortho)]
    ft = round(100 * log["train_acc"][end]; digits=2)
    fe = round(100 * log["test_acc"][end];  digits=2)
    fl = round(log["train_loss"][end]; digits=4)
    println("  [$tag]  Loss: $fl  |  Train: $ft%  |  Test: $fe%")

    jc = log["jacobian_conds"][end]
    if !isempty(jc)
        for (i, κ) in enumerate(jc)
            println("           Block $i Jacobian κ = $(round(κ; digits=2))")
        end
    end
    println()
end

println("="^70)
println("  Files saved in: $(OUT_DIR)/")
println("="^70)

## 11. Gradient Norm Comparison (Last Epoch)

Print side-by-side the gradient norms from the final training step of each model.  
Orthogonal init should show more uniform norms across layers (less vanishing).

In [ ]:
gn_x = log_xavier["grad_norms"][end]
gn_o = log_ortho["grad_norms"][end]

all_keys = sort(union(keys(gn_x), keys(gn_o)))

println(rpad("Parameter", 40) * rpad("Xavier ∥∇∥", 15) * "Ortho ∥∇∥")
println("-"^70)
for k in all_keys
    vx = haskey(gn_x, k) ? round(gn_x[k]; digits=6) : "—"
    vo = haskey(gn_o, k) ? round(gn_o[k]; digits=6) : "—"
    println(rpad(k, 40) * rpad(string(vx), 15) * string(vo))
end

## 12. Loss & Accuracy Curves

Simple text-based epoch log. For publication-quality plots, we'll add proper visualization in the analysis notebook later.

In [ ]:
println(rpad("Epoch", 8) * rpad("Xavier Loss", 14) * rpad("Xavier Test%", 14) *
       rpad("Ortho Loss", 14) * "Ortho Test%")
println("-"^64)
for i in 1:EPOCHS
    xl = round(log_xavier["train_loss"][i]; digits=4)
    xa = round(100*log_xavier["test_acc"][i]; digits=2)
    ol = round(log_ortho["train_loss"][i]; digits=4)
    oa = round(100*log_ortho["test_acc"][i]; digits=2)
    println(rpad(i, 8) * rpad(xl, 14) * rpad(xa, 14) * rpad(ol, 14) * oa)
end

## Next Steps

Once you're satisfied with the training results:

1. **Inference notebook** — load the saved `.jld2` weights, run predictions, compute saliency maps ($\partial\text{Output}/\partial\text{Input}$)
2. **Visualization** — Jacobian condition number plots, gradient flow heatmaps, saliency map comparison (Xavier vs Orthogonal)
3. **Paper figures** — publication-ready plots for the ECE452 paper